Лабораторная работа 1: CV

Импорт библиотек и скачивание данных

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import os

!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip -d intel_data

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
100% 346M/346M [00:01<00:00, 340MB/s]



Подготовка данных

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
train_path = 'intel_data/seg_train/seg_train'
test_path = 'intel_data/seg_test/seg_test'

transform_base = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_data = torchvision.datasets.ImageFolder(train_path, transform=transform_base)
test_data = torchvision.datasets.ImageFolder(test_path, transform=transform_base)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2)

classes = train_data.classes
print(f"Классы в датасете: {classes}")

Классы в датасете: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


Функции для обучения и оценки моделей

In [4]:
from tqdm import tqdm
import numpy as np

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    preds_all, labels_all = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item()
            preds = out.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            preds_all.append(preds.cpu())
            labels_all.append(y.cpu())

    preds_all = torch.cat(preds_all).numpy()
    labels_all = torch.cat(labels_all).numpy()

    f1 = f1_score(labels_all, preds_all, average='macro')

    return total_loss / len(loader), correct / total, f1

In [5]:
criterion = nn.CrossEntropyLoss()

In [6]:
# --- ResNet baseline
resnet_base = models.resnet18(weights='IMAGENET1K_V1')
resnet_base.fc = nn.Linear(resnet_base.fc.in_features, len(classes))
resnet_base = resnet_base.to(device)

opt_res_base = optim.Adam(resnet_base.parameters(), lr=1e-4)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s]


In [7]:
# --- ViT baseline
vit_base = models.vit_b_16(weights='IMAGENET1K_V1')
vit_base.heads.head = nn.Linear(vit_base.heads.head.in_features, len(classes))
vit_base = vit_base.to(device)

opt_vit_base = optim.Adam(vit_base.parameters(), lr=1e-5)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 194MB/s]


In [8]:
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, len(classes))
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cnn = MyCNN().to(device)
opt_cnn = optim.Adam(cnn.parameters(), lr=1e-4)

In [9]:
# =========================
# 9. TRAINING FUNCTION
# =========================
def run(model, loader, optimizer, name):
    print(f"\n===== {name} =====")

    for epoch in range(5):
        train_loss, train_acc = train_one_epoch(
            model, loader, optimizer, criterion
        )

        val_loss, acc, f1 = evaluate(model, test_loader, criterion)

        print(f"Epoch {epoch+1}: acc={acc:.4f}, f1={f1:.4f}")


Бейзлайны

In [10]:
run(resnet_base, train_loader, opt_res_base, "ResNet BASELINE")


===== ResNet BASELINE =====


100%|██████████| 220/220 [00:49<00:00,  4.44it/s]


Epoch 1: acc=0.9320, f1=0.9336


100%|██████████| 220/220 [00:50<00:00,  4.37it/s]


Epoch 2: acc=0.9300, f1=0.9311


...

Epoch 3: acc=0.9287, f1=0.9300


100%|██████████| 220/220 [00:49<00:00,  4.47it/s]


Epoch 4: acc=0.9177, f1=0.9182


100%|██████████| 220/220 [00:48<00:00,  4.52it/s]


Epoch 5: acc=0.9290, f1=0.9303


In [11]:
run(vit_base, train_loader, opt_vit_base, "ViT BASELINE")


===== ViT BASELINE =====


100%|██████████| 220/220 [08:22<00:00,  2.28s/it]


Epoch 1: acc=0.9333, f1=0.9347


100%|██████████| 220/220 [08:22<00:00,  2.29s/it]


Epoch 2: acc=0.9437, f1=0.9451


...

Epoch 3: acc=0.9453, f1=0.9466


100%|██████████| 220/220 [08:17<00:00,  2.26s/it]


Epoch 4: acc=0.9373, f1=0.9382


100%|██████████| 220/220 [08:17<00:00,  2.26s/it]


Epoch 5: acc=0.9437, f1=0.9450


In [12]:
run(cnn, train_loader, opt_cnn, "CNN BASELINE")


===== CNN BASELINE =====


100%|██████████| 220/220 [00:39<00:00,  5.58it/s]


Epoch 1: acc=0.4790, f1=0.4146


100%|██████████| 220/220 [00:38<00:00,  5.67it/s]


Epoch 2: acc=0.5237, f1=0.4959


...

Epoch 3: acc=0.5693, f1=0.5489


100%|██████████| 220/220 [00:38<00:00,  5.68it/s]


Epoch 4: acc=0.5900, f1=0.5690


100%|██████████| 220/220 [00:38<00:00,  5.69it/s]


Epoch 5: acc=0.6230, f1=0.6076


Гипотезы

In [13]:
# =========================
# 5. HYPOTHESIS 1 (AUGMENTATIONS)
# =========================
transform_h1 = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_loader_h1 = DataLoader(
    torchvision.datasets.ImageFolder(train_path, transform=transform_h1),
    batch_size=64,
    shuffle=True
)

resnet_h1 = models.resnet18(weights='IMAGENET1K_V1')
resnet_h1.fc = nn.Linear(resnet_h1.fc.in_features, len(classes))
resnet_h1 = resnet_h1.to(device)
opt_res_h1 = optim.Adam(resnet_h1.parameters(), lr=1e-4)

vit_h1 = models.vit_b_16(weights='IMAGENET1K_V1')
vit_h1.heads.head = nn.Linear(vit_h1.heads.head.in_features, len(classes))
vit_h1 = vit_h1.to(device)
opt_vit_h1 = optim.Adam(vit_h1.parameters(), lr=1e-5)

cnn_h1 = MyCNN().to(device)
opt_cnn_h1 = optim.Adam(cnn_h1.parameters(), lr=1e-4)

In [14]:
# =========================
# 6. HYPOTHESIS 2 (LOW LR)
# =========================
resnet_h2 = models.resnet18(weights='IMAGENET1K_V1')
resnet_h2.fc = nn.Linear(resnet_h2.fc.in_features, len(classes))
resnet_h2 = resnet_h2.to(device)
opt_res_h2 = optim.Adam(resnet_h2.parameters(), lr=5e-5)

vit_h2 = models.vit_b_16(weights='IMAGENET1K_V1')
vit_h2.heads.head = nn.Linear(vit_h2.heads.head.in_features, len(classes))
vit_h2 = vit_h2.to(device)
opt_vit_h2 = optim.Adam(vit_h2.parameters(), lr=5e-6)

cnn_h2 = MyCNN().to(device)
opt_cnn_h2 = optim.Adam(cnn_h2.parameters(), lr=5e-5)

In [15]:
# =========================
# 7. HYPOTHESIS 3 (ADAMW)
# =========================
resnet_h3 = models.resnet18(weights='IMAGENET1K_V1')
resnet_h3.fc = nn.Linear(resnet_h3.fc.in_features, len(classes))
resnet_h3 = resnet_h3.to(device)
opt_res_h3 = optim.AdamW(resnet_h3.parameters(), lr=1e-4, weight_decay=1e-4)

vit_h3 = models.vit_b_16(weights='IMAGENET1K_V1')
vit_h3.heads.head = nn.Linear(vit_h3.heads.head.in_features, len(classes))
vit_h3 = vit_h3.to(device)
opt_vit_h3 = optim.AdamW(vit_h3.parameters(), lr=1e-5, weight_decay=1e-4)

cnn_h3 = MyCNN().to(device)
opt_cnn_h3 = optim.AdamW(cnn_h3.parameters(), lr=1e-4, weight_decay=1e-4)

Проверка первой гипотезы

In [16]:
# ResNet H1
run(resnet_h1, train_loader_h1, opt_res_h1, "ResNet H1")


===== ResNet H1 =====


100%|██████████| 220/220 [01:45<00:00,  2.08it/s]


Epoch 1: acc=0.9300, f1=0.9312


100%|██████████| 220/220 [01:45<00:00,  2.08it/s]


Epoch 2: acc=0.9293, f1=0.9305


...

Epoch 3: acc=0.9343, f1=0.9354


100%|██████████| 220/220 [01:46<00:00,  2.06it/s]


Epoch 4: acc=0.9220, f1=0.9233


100%|██████████| 220/220 [01:46<00:00,  2.06it/s]


Epoch 5: acc=0.9270, f1=0.9281


In [17]:
# ViT H1
run(vit_h1, train_loader_h1, opt_vit_h1, "ViT H1")


===== ViT H1 =====


100%|██████████| 220/220 [09:23<00:00,  2.56s/it]


Epoch 1: acc=0.9347, f1=0.9357


100%|██████████| 220/220 [09:23<00:00,  2.56s/it]


Epoch 2: acc=0.9460, f1=0.9472


...

Epoch 3: acc=0.9470, f1=0.9483


100%|██████████| 220/220 [09:23<00:00,  2.56s/it]


Epoch 4: acc=0.9477, f1=0.9487


100%|██████████| 220/220 [09:08<00:00,  2.49s/it]


Epoch 5: acc=0.9460, f1=0.9474


In [18]:
# CNN H1
run(cnn_h1, train_loader_h1, opt_cnn_h1, "CNN H1")


===== CNN H1 =====


100%|██████████| 220/220 [01:28<00:00,  2.49it/s]


Epoch 1: acc=0.4733, f1=0.4144


100%|██████████| 220/220 [01:28<00:00,  2.50it/s]


Epoch 2: acc=0.5440, f1=0.5259


...

Epoch 3: acc=0.5610, f1=0.5421


100%|██████████| 220/220 [01:28<00:00,  2.47it/s]


Epoch 4: acc=0.5840, f1=0.5723


100%|██████████| 220/220 [01:28<00:00,  2.48it/s]


Epoch 5: acc=0.6053, f1=0.5980


Проверка второй гипотезы

In [ ]:
run(resnet_h2, train_loader, opt_res_h2, "ResNet H2")


===== ResNet H2 =====


100%|██████████| 220/220 [00:47<00:00,  4.59it/s]


Epoch 1: acc=0.9247, f1=0.9256


100%|██████████| 220/220 [00:47<00:00,  4.59it/s]


In [ ]:
run(vit_h2, train_loader, opt_vit_h2, "ViT H2")

In [ ]:
run(cnn_h2, train_loader, opt_cnn_h2, "CNN H2")

Проверка третьей гипотезы

In [ ]:
run(resnet_h3, train_loader, opt_res_h3, "ResNet H3")

In [ ]:
run(vit_h3, train_loader, opt_vit_h3, "ViT H3")

In [ ]:
run(cnn_h3, train_loader, opt_cnn_h3, "CNN H3")

Результаты

In [ ]:
print("\n📊 Итоговое сравнение моделей\n")

print(f"{'Модель':<15} | {'Accuracy':<10} | {'F1-score':<10}")
print("-" * 50)

for name, (acc, f1) in results.items():
    print(f"{name:<15} | {acc:<10.4f} | {f1:<10.4f}")